# Topic 1 — Exact Diagonalization

## 1. Tensor product and basis states

We consider the Hilbert space of $n$ qubits (equivalently, $n$ spin-$\frac{1}{2}$ degrees of freedom). For a single qubit, the Hilbert space is two-dimensional, spanned by the basis states $|0\rangle$ and $|1\rangle$. An arbitrary qubit state can be written as 
$$
    |\psi\rangle = \alpha\,|0\rangle + \beta\,|1\rangle~,
$$ 
where $|\alpha|^2 + |\beta|^2 = 1$. (For a spin-$\frac{1}{2}$ particle, one identifies $|1\rangle \equiv |\!\downarrow\rangle$ and $|0\rangle \equiv |\!\uparrow\rangle$.)

Once a basis is chosen, we store the state numerically as an array of its coefficients — for the single-qubit case: `psi = np.array([alpha, beta])`, where the $j$-th element stores the coefficient of the basis state $|j\rangle$.

The total Hilbert space of $n$ qubits is the *tensor product* of $n$ single-qubit spaces (qubits labeled $j = 0, \ldots, n-1$), with dimension $2^n$. An orthonormal basis is the **computational basis**
$$|\sigma_{n-1}\,  \cdots\, \sigma_1\, \sigma_{0}\rangle, \qquad \sigma_j \in \{0, 1\}~.$$
A general $n$-qubit state requires $2^n$ complex coefficients to be fully specified:
$$|\psi\rangle = \sum_{\sigma_0 \cdots \sigma_{n-1}} a_{\sigma_{n-1}  \cdots  \sigma_{0}}|\sigma_{n-1}  \cdots  \sigma_{0}\rangle~.$$

A $2^n$-element array is indexed by integers `ind` $\in \{0, \ldots, 2^n - 1\}$; we therefore need a rule to map each index to a basis bitstring.
To this end, we use the binary representation of an integer
$$\text{ind} = \sum_{j=0}^{n-1} 2^{\,j}\,\sigma_j = (\sigma_{n-1}\,\cdots\,\sigma_1\sigma_0)_2~,$$
and the two conventions differ in how the ket is ordered:

---
**(Big-endian):** 
$$\quad \text{ind} = \displaystyle\sum_{j=0}^{n-1} 2^{\,j}\,\sigma_{n-j-1} \;\Longleftrightarrow\; |\sigma_0\,\sigma_1\,\cdots\,\sigma_{n-1}\rangle$$
Written in standard binary, `ind` reads $\sigma_{n-1}\cdots\sigma_1\sigma_0$ — the *reverse* of the ket ordering. Anecdotally, most physicists seem to use this convention. I guess mainly it's because if you have a bit string `bstring = '0101'`, then `bstring[0] = 0`, the qubit label matches the array/string index.

For example, for $n = 2$:

| index | bitstring | basis |
|:---:|:---:|:---:|
| 0 | `'00'` | $\|0_00_1\rangle$ |
| 1 | `'01'` | $\|0_01_1\rangle$ |
| 2 | `'10'` | $\|1_00_1\rangle$ |
| 3 | `'11'` | $\|1_01_1\rangle$ |

---
**(Little-endian):** The leftmost position in the ket carries the most significant bit.
$$\quad \text{ind} = \displaystyle\sum_{j=0}^{n-1} 2^{\,j}\,\sigma_j \;\Longleftrightarrow\; |\sigma_{n-1}\,\cdots\,\sigma_1\,\sigma_0\rangle~.$$
Most computer scientists seem to use this convension since the bit ordering matches with the binary representation. The qiskit package from IBM uses this convention, and is **adopted throughout this course**.

### Exercise 1:
Suppose qubit $0$ is $|\psi\rangle_0 = a_0 |0\rangle_0 + a_1|1\rangle_0$, qubit $1$ is $|\chi\rangle_1 = b_0 |0\rangle_1 + b_1|1\rangle_1$, then the quantum state of this $n=2$ qubit system is
$$
|\psi'\rangle = |\chi\rangle_1 \otimes |\psi\rangle_0 = a_0b_0|00\rangle + b_0a_1|01\rangle +b_1a_0|10\rangle + a_1b_1|11\rangle.
$$

1. Construct a function `tensorprod(psi, chi)` using numpy's built-in function `np.kron` so that the output state is consistent with our endianness convention. (Hint: try `np.kron(psi, chi)` and `np.kron(psi, chi)`. Which one gives the result corresponding to our convention?)

2. Pick two testing states `psi` and `chi` that are able to demonstrate your function output results consistent with our convention.

In [ ]:
import numpy as np
# 1. construct the tensor product of two qubit states: 
def tensorprod(psi, chi):
    pass


In [ ]:
# 2. Pick two qubit states to demonstrate the tensor product function works as intended.


## 2. Operators and Hamiltonian

### 2.1 Operators
An operator is a linear map of quantum states, which can be fully specified by its action on all the basis states. For example, the Pauli operators $X$, $Y$ and $Z$ are defined as
$$
\begin{aligned}
X|0\rangle &= |1\rangle~, \quad X|1\rangle = |0\rangle~,\\
Y|0\rangle &= i|1\rangle~, \quad Y|1\rangle = -i|0\rangle~ (Y|\sigma\rangle = i(-1)^{\sigma}|\bar{\sigma}\rangle,\ \text{i.e. } Y = iXZ),\\
Z|0\rangle &= |0\rangle~, \quad Z|1\rangle = -|1\rangle~ (Z|\sigma\rangle = (-1)^{\sigma}|\sigma\rangle).
\end{aligned}
$$
(These are precisely the spin-$\frac{1}{2}$ operators you learned without the $\hbar/2$ factor.)
We also include the identity operator $I$, where $I|\sigma\rangle = |\sigma\rangle$ here.
We also use $|\pm\rangle = \frac{1}{\sqrt{2}}(|0\rangle \pm |1\rangle)$ to denote the eigenbasis of $X$.

Just like the quantum states, once you pick an orthonormal basis, you can write an operator as
$$
\hat{O} = \sum_{\sigma,\sigma'} O_{\sigma,\sigma'} |\sigma_{n-1} \cdots \sigma_0\rangle \langle \sigma_{n-1}' \cdots \sigma_0'|~,
$$
and on a computer program, we use a $2^n \times 2^n$ matrix `Omat` to store the coefficients $O_{\sigma,\sigma'}$. The resulting quantum state from the operator $\hat{O}$ acting on a state $|\psi\rangle$, or $\hat{O}|\psi\rangle$, therefore has coefficients given by the matrix-vector product `Omat @ psi`.

A particular special kind of operator, dubbed "Pauli-string" operator, is of the type composed of tensor product of $I$ and Pauli operators, e.g. $I_0 \otimes X_1 \otimes Y_2 \otimes I_3 \cdots \otimes Z_{n-1}$, where the subscript denotes the qubit the corresponding operator acts on. We often omit the tensor product $\otimes$ and write it as $I_0  X_1  Y_2  I_3 \cdots  Z_{n-1}$, or identify the location of the operator according to its string location $Z \cdots IYXI$ (see why physicists usually use the other endianess convention......because if you have a string `P='XYZ'`, `P[0]` gives you 'X'),  hence the name "Pauli-string" (analogous to bitstring). Furthermore, we also abbreviate it further by skipping the identities, e.g. $X_1Y_2Z_{n-1}$.
(Q: under our convention, the matrix representation of $X_0Y_1$ for two qubits should be
`np.kron(X,Y)` or `np.kron(Y,X)`?)

Pauli strings have several useful properties, especially when we discuss Heisenberg-picture evolution. One nice property is that they form an **orthonormal basis for the space of linear operators**. Since linear operators themselves form a vector space, we can define an inner product between two operators $A$ and $B$ as

$$
\langle A,B\rangle = \frac{\mathrm{Tr}[A^\dagger B]}{\mathrm{Tr}[I]}.
$$

For an $n$-qubit system, $\mathrm{Tr}[I]=2^n$. One can then show that Pauli strings satisfy

$$
\frac{\mathrm{Tr}[P_s^\dagger P_{s'}]}{\mathrm{Tr}[I]}
=
\delta_{s,s'}.
$$

**Q:** Show this. 

### 2.2 Hamiltonian

A Hamiltonian generates the quantum dynamics according to the Schrödinger equation. For a time-independent Hamiltonian $H$ and an initial quantum state $|\psi\rangle$, one can solve the Schrödinger equation
$$
i\frac{\partial}{\partial t}|\psi\rangle = \hat{H} |\psi\rangle
$$
with
$$
|\psi(t)\rangle = e^{-iHt} |\psi\rangle~.
$$
Evidently, the operator $U(t) = e^{-iHt}$ has a matrix representation given as the matrix exponentiation of the matrix representation of the Hamiltonian `Hmat`. (We would call it Hamiltonian matrix or just simply Hamiltonian, if the context is clear.) If one is able to construct the matrix $U(t) = e^{-iHt}$, then one can obtain $|\psi(t)\rangle$ via matrix-vector multiplication.

Here are some famous examples of Hamiltonians:

**Quantum Ising model:**
$$
H = -\sum_{i<j} J_{ij}Z_iZ_j - \sum_{j} h_j X_j
$$

**Quantum Heisenberg model:**
$$
H = -\sum_{i<j} J_{ij}(X_iX_j+Y_iY_j+Z_iZ_j) - \sum_{j} h_j X_j
$$

We will first use a "quick-and-dirty" method to construct a Hamiltonian in Exercise 2.

### 2.3 Diagonalization and matrix exponentiation
Once the matrix representation `Hmat` is constructed, one can obtain its eigenvalues and eigenvectors by calling `energies, eigvecs = np.linalg.eigh(Hmat)`, where `eigvecs[:, k]` is the $k$-th eigenvector (i.e., `eigvecs` is the unitary $U$ in the decomposition below). The eigendecomposition gives
$$
H = U D U^{\dagger}~,
$$
where $D$ is the diagonal matrix of eigenvalues $\epsilon_k$.

The matrix exponential then follows as
$$
e^{-iHt} = U\, e^{-iDt}\, U^{\dagger}~,
$$
where $e^{-iDt}$ is diagonal with entries $e^{-i\epsilon_k t}$. This is the preferred approach when evolving over many time steps: diagonalize once, then evaluate cheaply for each $t$. Alternatively, one can call `scipy.linalg.expm(-1j * Hmat * t)` directly for a single time step.

### Exercise 2: Quick-and-dirty method

1. For the "quick-and-dirty" method of constructing the Hamiltonian matrix, build a function `paulistring_to_matrix(paulistring, qubitlabel, n_qubits)` that takes a Pauli string `paulistring`, its associated qubit labels `qubitlabel`, and the total number of qubits `n_qubits` as inputs, and returns the corresponding $2^n \times 2^n$ matrix. For example, the matrix for $X_2 Y_3 Z_6$ on a 10-qubit system is obtained with `paulistring_to_matrix('XYZ', [2, 3, 6], 10)`. (Hint: use `np.kron` and be careful about the endianness convention.)

    Pick a set of `paulistring, qubitlabel, n_qubits` so human can verify easily that the function works correctly and print the result.

2. Use the above function to construct the 1D transverse-field Ising Hamiltonian with periodic boundary conditions for `n_qubits = 10` qubits, with parameters $J = 1.0$ and $h = 0.5$:
$$
H = -J \sum_{j=0}^{n-1} X_j X_{j+1} - h \sum_{j=0}^{n-1} Z_j~,
$$
where we identify $j + n \equiv j$ (periodic boundary).

3. Diagonalize the Hamiltonian and print the ground state energy. (Note: the 1D transverse-field Ising model with periodic boundary conditions is exactly solvable via the Jordan–Wigner transformation.)

In [ ]:
import numpy as np

# --- Pauli matrices ---
# _I = np.eye(2, dtype=complex)
# _X = np.array([[0, 1], [1, 0]], dtype=complex)
# _Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
# _Z = np.array([[1, 0], [0, -1]], dtype=complex)
# _PAULI = {'I': _I, 'X': _X, 'Y': _Y, 'Z': _Z}


def paulistring_to_matrix(paulistring, qubitlabel, n_qubits):
    pass




[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  0.+0.j -0.+0.j]
 [ 0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j -0.+0.j  0.+0.j -1.+0.j]]
Ground state energy: -10.63560441


In [ ]:

# --- 2. Build 10-qubit transverse-field Ising Hamiltonian with PBC ---

# --- 3. Diagonalize ---


## 3. Hash list for the Hilbert space

### 3.1 Exploiting symmetry and conservation laws

As we can see, the bottleneck of exact diagonalization is its exponential resource requirement $2^n$. However, for many problems there are **symmetries** and **conservation laws** one can exploit. For example, for the Heisenberg model we saw previously, the total magnetization $M = \sum_{j=0}^{n-1} Z_j$ is a conserved quantity $[H, M] = 0$. This means that the Hamiltonian can be written in block-diagonal form where each block is labeled by the magnetization quantum number $m = -n,\, -n+2,\, \ldots,\, n-2,\, n$. The Hilbert-space dimension within each sector is reduced from $2^n$ to $\binom{n}{(n+m)/2}$.
For the quantum Ising model, the parity operator $P = \prod_{j=0}^{n-1} Z_j$ is a conserved quantity $[H,P]=0$

However, this also means that when constructing the matrix representation of the Hamiltonian within a given magnetization sector, we need to know which bitstring each row/column index corresponds to. One can compute this on-the-fly or prepare a look-up table. Here we use a **hash map** (the `dict` data structure in Python) to store this look-up table.

In the following exercise, we construct this look-up table for the case where each sector basis state is a single computational basis state — as in the magnetization sector above.

### References 
See [this](https://physics.bu.edu/~sandvik/vietri/vietri.pdf) and the attached file (LesHouches_ED_Lecture.pdf) for the extension to more general symmetries, such as lattice translation symmetry, where sector basis states are linear combinations of computational basis states.




### Exercise 3: Construction of a Look-up Table

1. Construct a function `build_basis_table` that takes `n_qubits` (number of qubits) and a filter function `basis_filter=None` (defaulting to `None`), and returns two `dict`s: one mapping each sector index to the corresponding computational basis bitstring `bstring` for which `basis_filter(bstring)` returns `True`, and the other mapping each basis bitstring to its corresponding index. When `basis_filter` is `None`, include all $2^n$ states.

2. Construct a function `parity_filter(p)` that takes a target parity eigenvalue $p = \pm 1$ and **returns a `basis_filter`-compatible callable** (i.e., `bitstring → bool`) selecting bitstrings whose parity satisfies $p = (-1)^{n_\downarrow}$ ($n_\downarrow$ = number of up-spins, i.e. `'1'` bits).

3. Construct a function `magnetization_filter(m)` that takes a target magnetization number $m$ and **returns a `basis_filter`-compatible callable** selecting bitstrings satisfying $M = \sum_j Z_j = 2n_\downarrow - n = m$.

4. Use the two filter functions to generate the look-up tables for the $p = +1$ parity sector and the $m = 0$ magnetization sector, both for $n = 10$. Verify your results: what should the sizes of these tables equal in terms of $n$?

In [ ]:
import math

def build_basis_table(n_qubits, basis_filter=None):
    pass

def parity_filter(p):
    pass

def magnetization_filter(m):
    pass

# --- Verification for n = 10 ---
# p=+1 parity sector: half of all states → 2^(n-1)
# m=0 magnetization sector: n/2 spins up → C(n, n/2)
n = 10


p=+1 sector: 512 states  (expected 2^(n-1) = 512)
m=0  sector: 252 states  (expected C(n,n/2) = 252)


## 4. Linear operator and matrix representation

### 4.1 Matrix representation

Now that one considers only a subset of the basis, the "quick-and-dirty" tensor-product method will no longer work.

To this end, it is useful to recall some more abstract aspects of linear algebra. In quantum mechanics, the Pauli strings and Hamiltonian are operators, which map a quantum state to a quantum state $|\chi \rangle = \hat{O} |\psi \rangle$. In the linear algebra language, these operators are "linear operators", whose action can be described by how the operator $\hat{O}$ maps each basis state $|a\rangle$ for $a = 0, \cdots 2^n -1$. Once this is specified, the mapping of any vector can be calculated using the basis expansion $|\psi\rangle = \sum_{a = 0}^{2^n -1} \psi_a |a\rangle$, and $\hat{O}|\psi\rangle = \sum_{a = 0}^{2^n -1} \psi_a (\hat{O}|a\rangle)$. 

In a computer program, one can define a function which takes a vector as an input, and outputs the corresponding mapped vector, if such a mapping is structured enough to be hardcoded. This function is commonly called **Abstract Matrix** or **Matrix-free operator**, and has the advantage that you never need to use any memory to store the matrix representation of the operator. 

On the other hand, using this mapping, one can also construct the matrix representation of $\hat{O}$ with matrix elements $[O]_{ba} = \langle b | \hat{O} |a\rangle$ by calculating $|v_a \rangle = \hat{O}|a\rangle $, and putting the coefficients of $|v_a \rangle$ in the $a$-th column of the matrix $[O]_{ba}$. That is, $|v_a \rangle = \sum_{b} |b\rangle[O]_{ba} $.

### 4.2 Pauli-string operators

The Pauli-string operators $P$ play a special role for operators in qubit systems. 
1. They form an "operator" basis for the linear operators. That is, any linear operator can be expanded using the Pauli-string operators $\hat{O} = \sum_{a} c_a P_a$ with some coefficients $c_a$. (Q: What special property does $c_a$ have if $\hat{O}$ is Hermitian?)
2. The Pauli-string operator maps a computational basis state to a computational basis state. 
$$
\begin{aligned}
I|\sigma\rangle &= |\sigma\rangle, & &\text{coefficient } +1 \\
X|\sigma\rangle &= |\bar\sigma\rangle, & &\text{coefficient } +1 \\
Y|\sigma\rangle &= i(-1)^{\sigma}|\bar\sigma\rangle, & &\text{coefficient } i(-1)^{\sigma} \\
Z|\sigma\rangle &= (-1)^{\sigma}|\sigma\rangle, & &\text{coefficient } (-1)^{\sigma}
\end{aligned}
$$
where $\bar\sigma = 1 - \sigma$ is the bit flip. A Pauli string $P = P_{n-1} \otimes \cdots\otimes P_1   \otimes P_0$ therefore maps each basis state $|s\rangle$ to a *single* basis state $|s'\rangle$:
$$
P|s\rangle = \alpha(s)\,|s'\rangle~,
$$
where $s'$ is obtained from $s$ by **flipping the bits at all $X$ and $Y$ sites**, and $c(s)$ is the **product of the single-qubit factors** evaluated at each bit of $s$.

Computing this requires only bitstring manipulation — no matrix construction.

This also means that, for any operator $\hat{O} = \sum_{a} c_a P_a$ where the number of nonzero $c_a$ is of order $n$ instead of $4^n$, then we know $|v_a \rangle = \hat{O}|a\rangle $ will have of order $n$ nonzero coefficients, so the matrix representation of $\hat{O}$ will have $\sim n \times 2^n$ nonzero entries --- called a **sparse matrix** as opposed to a dense matrix which has $\sim 2^n \times 2^n$ nonzero entries. This is a property one can exploit to construct the matrix representation more efficiently and store it in a more memory-efficient way.
Note that usually the Hamiltonians we are interested in have this property.


### Exercise 4:

1. Construct a function `apply_paulistring(paulis, qubit_labels, bitstring)` that takes a Pauli string, represented as a tuple of an array of `'X'`, `'Y'`, `'Z'` characters and an array of its corresponding qubit labels, and a bitstring $s$, and outputs the new bitstring $s'$ and its phase $\alpha$.

2. Use the above function and the `build_basis_table` function to construct the transverse quantum Ising Hamiltonian with PBC in both parity sectors $p = +1$ and $p = -1$ for $n = 10$ qubits, with parameters $J = 1.0$ and $h = 0.5$:
$$
H = -J \sum_{j=0}^{n-1} X_j X_{j+1} - h \sum_{j=0}^{n-1} Z_j~,
$$
where we identify $j + n \equiv j$ (periodic boundary).
Print the ground state energies from both sectors. Which sector does the true ground state live in? How close they are? How about $h = 1.5$? How does it compare to the "quick-and-dirty" result previously?

In [ ]:
import numpy as np

def apply_paulistring(paulis, qubit_labels, bitstring):
    pass


def build_sector_hamiltonian(n_qubits, ham_terms, basis_filter=None):
    pass


# --- Hamiltonian terms: H = -J Σ X_j X_{j+1} - h Σ Z_j ---
# Parity operator: P = Π Z_j, eigenvalue p = (-1)^(n_up).
# Both XX and Z terms preserve n_up mod 2, so each parity sector is invariant.


p=+1 sector:  dim = 512,  E_0 = -10.63560441
p=-1 sector:  dim = 512,  E_0 = -10.63528368
True ground state energy: -10.63560441
Quick-and-dirty result:   -10.63560441  (same spectrum, different basis rotation)


## 5. Fermionic systems

In condensed matter physics and quantum chemistry, we often have to deal with many electrons, which are fermions. It is often easier to formulate such problems in the **second-quantized** language.

Suppose we have $n$ fermionic modes labeled by $j=0,\ldots,n-1$. The Hilbert space (Fock space) is spanned by the occupation-number states

$$
    |m_0 \ldots m_{n-1}\rangle, \qquad m_j=0,1.
$$

(Sorry, here I use a different endianness convention. This section is a bit of a digression/bonus, so it should be fine.)

This looks awfully similar to a system of $n$ qubits. However, the difference lies in the definition of the fermionic operators. For each mode, we have a creation operator $c_j^\dagger$ and an annihilation operator $c_j$. Unlike operators acting on different qubits, fermionic operators satisfy the anticommutation relations

$$
\{c_i,c_j\}=0, \qquad
\{c_i^\dagger,c_j^\dagger\}=0, \qquad
\{c_i^\dagger,c_j\}=\delta_{ij}.
$$

For example, the two-particle state $|1100\ldots\rangle$ can be obtained from the vacuum $|00\ldots0\rangle$ by applying $c_0^\dagger$ and $c_1^\dagger$. However, applying them in the opposite order gives an extra minus sign because

$$
c_0^\dagger c_1^\dagger = -c_1^\dagger c_0^\dagger.
$$

We therefore need to choose a sign convention for the occupation-number basis. We will use

$$
    |m_0 \ldots m_{n-1}\rangle
    =
    (c_0^\dagger)^{m_0}
    \cdots
    (c_{n-1}^\dagger)^{m_{n-1}}
    |00\ldots0\rangle,
    \qquad m_j=0,1.
$$

You can then show that

$$
\begin{aligned}
c_j |m_0,\ldots,m_j=0,\ldots,m_{n-1}\rangle
&=0,\\
c_j |m_0,\ldots,m_j=1,\ldots,m_{n-1}\rangle
&=(-1)^{\sum_{k<j}m_k}
|m_0,\ldots,m_j=0,\ldots,m_{n-1}\rangle,\\
c_j^\dagger |m_0,\ldots,m_j=1,\ldots,m_{n-1}\rangle
&=0,\\
c_j^\dagger |m_0,\ldots,m_j=0,\ldots,m_{n-1}\rangle
&=(-1)^{\sum_{k<j}m_k}
|m_0,\ldots,m_j=1,\ldots,m_{n-1}\rangle.
\end{aligned}
$$

One should therefore be able to construct the matrix representations of $c_j$ and $c_j^\dagger$ directly from these rules.

Notice that even though $c_j$ and $c_j^\dagger$ appear to act locally on mode $j$, their action depends on the occupation of **all modes $k<j$**, which produces the phase factor

$$
(-1)^{\sum_{k<j}m_k}.
$$

In the qubit language, the operator $Z_k$ gives precisely such a phase:

$$
Z_k|m_k\rangle = (-1)^{m_k}|m_k\rangle.
$$

Also,

$$
\frac{X+iY}{2}=|0\rangle\langle1|,
\qquad
\frac{X-iY}{2}=|1\rangle\langle0|.
$$

We can therefore identify

$$
\begin{aligned}
c_j
&\rightarrow
\left(\prod_{k<j} Z_k\right)
\frac{X_j+iY_j}{2},\\
c_j^\dagger
&\rightarrow
\left(\prod_{k<j} Z_k\right)
\frac{X_j-iY_j}{2}.
\end{aligned}
$$

This is the famous **Jordan--Wigner transformation**.

When constructing the matrix representation of a fermionic Hamiltonian in the occupation-number basis, you are essentially already keeping track of these fermionic signs. The Jordan--Wigner transformation makes this correspondence explicit by mapping fermionic operators into qubit operators (Pauli strings). This allows us to represent a fermionic problem as a spin/qubit problem and apply many simulation techniques developed for spin systems. It also provides a direct way to simulate fermionic systems on a quantum computer.

However, there is a price to pay. For example, a hopping term

$$
c_i^\dagger c_j + \mathrm{h.c.}
$$

can map to relatively high-weight Pauli strings when $i$ and $j$ are far apart. For example, for $i<j$, the resulting Pauli strings contain a string of $Z$ operators between sites $i$ and $j$.

This is often manageable for classical algorithms, but it can be inconvenient for quantum simulation because implementing high-weight Pauli rotations generally requires deeper quantum circuits.

There are therefore other fermion-to-qubit encodings that map the fermionic Fock states

$$
|m_0\ldots m_{n-1}\rangle
$$

to the qubit computational basis in different ways, with the goal of giving the corresponding fermionic operators more desirable Pauli weights. One famous example is the **Bravyi--Kitaev transformation**.